# Phase 4 — QLoRA fine-tune of the chess coach (v1)

Fine-tunes **Qwen2.5-3B-Instruct** on our `v1` distillation dataset (104 grounded
`facts → coaching` pairs) using **QLoRA**: a 4-bit frozen base + trainable LoRA adapters.

**Goal:** beat the honest baseline on the 29-game held-out set — **factual 2.90 / overall 3.38**.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. The whole run fits in ~6 GB of VRAM.

Pipeline: load 4-bit base → attach LoRA → SFT (completion-only loss) → save adapter →
sanity-check → merge → export GGUF for Ollama → (back on the Mac) `eval.run`.

## 0 · Confirm the GPU
If this fails, switch the runtime to a T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=memory.total,memory.used --format=csv

## 1 · Install
Pinned to a coherent stack. If Colab's preinstalled torch complains, we'll adjust versions
together — library churn here is normal.

In [ ]:
%pip install -q -U \
  "transformers==4.44.2" "trl==0.9.6" "peft==0.12.0" \
  "bitsandbytes==0.43.3" "accelerate==0.33.0" "datasets==2.21.0"

## 2 · Upload the dataset
Upload **`train.jsonl`** and **`val.jsonl`** from `training/datasets/v1/` in the repo
(they're gitignored, so they live on your machine, not GitHub).

In [ ]:
from google.colab import files
print("Select train.jsonl and val.jsonl …")
up = files.upload()
import json
for f in ("train.jsonl", "val.jsonl"):
    n = sum(1 for _ in open(f))
    print(f"{f}: {n} examples")

## 3 · Load Qwen2.5-3B in 4-bit  (the QLoRA base)

`BitsAndBytesConfig` is Leg 4 in code:
- `nf4` — NormalFloat-4, the quantile-based 4-bit that suits ~normal weights
- `double_quant` — quantize the block scales too (~0.4 bit/param saved)
- `compute_dtype=bf16` — 4-bit **storage**, 16-bit **math** (dequantized per matmul)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL = "Qwen/Qwen2.5-3B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
# make the 4-bit model trainable-through: cast norms to fp32, enable grad checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False          # incompatible with gradient checkpointing
print(model.get_memory_footprint() / 1e9, "GB resident (4-bit base)")

## 4 · Attach LoRA adapters  (Leg 3, our knobs)

Only these tiny `A`/`B` matrices train; the 3B base stays frozen. `print_trainable_parameters`
should report **< 1%** trainable.

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16,                 # rank — capacity; modest, since we only have ~82 train examples
    lora_alpha=32,        # scale = alpha/r; decouples strength from rank
    lora_dropout=0.05,    # regularize against overfitting a tiny set
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],   # all linear layers
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5 · Format data + completion-only loss

Each row is `{"messages":[system, user, assistant]}`. We render it with Qwen's chat template,
then use `DataCollatorForCompletionOnlyLM` so the loss is computed **only on the assistant's
coaching** — the model isn't trained to reproduce the facts prompt, only to *respond* to it.

In [ ]:
from datasets import load_dataset
from trl import DataCollatorForCompletionOnlyLM

ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def render(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}
ds = ds.map(render, remove_columns=ds["train"].column_names)
print(ds["train"][0]["text"][:500], "\n…")

# Qwen marks the assistant turn with this template; everything before it is masked from loss.
collator = DataCollatorForCompletionOnlyLM(
    response_template="<|im_start|>assistant\n", tokenizer=tok,
)

## 6 · Train

Hyperparameters, annotated:
- **3 epochs** — small set; a few passes is enough, more risks overfitting (watch eval loss).
- **lr 2e-4, cosine** — the standard LoRA learning rate.
- **effective batch 16** (`2 × 8` grad-accum) — fits T4 memory while keeping updates stable.
- **`paged_adamw_8bit`** — 8-bit paged optimizer; offloads state to CPU on spikes.

Watch **train vs eval loss**: if eval loss turns up while train keeps falling, we're overfitting —
lower epochs or `r`.

> Note: transformers 4.44 uses `evaluation_strategy`; newer versions renamed it `eval_strategy`.

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qwen-coach-lora-v1",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,           # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,                           # required for completion-only masking
    report_to="none",
    seed=0,
)
trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    tokenizer=tok, data_collator=collator,
)
trainer.train()

## 7 · Save the adapter
Just the `A`/`B` matrices — a few hundred MB, not the whole model.

In [ ]:
trainer.save_model("qwen-coach-lora-v1")
tok.save_pretrained("qwen-coach-lora-v1")
!du -sh qwen-coach-lora-v1
!zip -qr qwen-coach-lora-v1.zip qwen-coach-lora-v1
files.download("qwen-coach-lora-v1.zip")   # keep a copy locally

## 8 · Sanity-check before merging

Run the fine-tuned model on a real val prompt (system + facts, **no** gold answer) and eyeball
the coaching. The thing to look for: does it stay **grounded** — cite only moves in the facts,
call a clean game clean — rather than inventing moves like the baseline did?

In [ ]:
import json
model.eval()
sample = json.loads(open("val.jsonl").readline())["messages"]
prompt = [m for m in sample if m["role"] in ("system", "user")]
ids = tok.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=300, do_sample=False)
print("FACTS:\n", prompt[-1]["content"][:400])
print("\nCOACH (fine-tuned):\n", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 9 · Merge → export GGUF for Ollama

To score with `eval.run` (which drives Ollama), we merge the adapter into the base, convert to
**GGUF**, and quantize to `q4_k_m`. This is the heaviest/most version-sensitive step — building
llama.cpp takes a few minutes, and we may need to iterate.

> Free the 4-bit model first to avoid OOM (merging loads the base in fp16).

In [ ]:
del model, trainer
torch.cuda.empty_cache()

from peft import AutoPeftModelForCausalLM
merged = AutoPeftModelForCausalLM.from_pretrained(
    "qwen-coach-lora-v1", torch_dtype=torch.float16, device_map="auto",
).merge_and_unload()
merged.save_pretrained("qwen-coach-merged", safe_serialization=True)
tok.save_pretrained("qwen-coach-merged")
print("merged fp16 model saved")

In [ ]:
# HF -> GGUF (f16), then quantize to q4_k_m
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py qwen-coach-merged \
    --outfile qwen-coach-v1-f16.gguf --outtype f16
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release >/dev/null && cmake --build build -j --target llama-quantize >/dev/null
!./llama.cpp/build/bin/llama-quantize qwen-coach-v1-f16.gguf qwen-coach-v1-q4_k_m.gguf q4_k_m
!ls -lh qwen-coach-v1-q4_k_m.gguf
files.download("qwen-coach-v1-q4_k_m.gguf")

## 10 · Back on the Mac — register with Ollama and score

Once `qwen-coach-v1-q4_k_m.gguf` is downloaded, on your machine:

```bash
# Modelfile
cat > Modelfile <<'EOF'
FROM ./qwen-coach-v1-q4_k_m.gguf
EOF

ollama create qwen-coach-v1 -f Modelfile

# score on the 29-game held-out set with the same Opus judge as the baseline
python -m eval.run --coach-model qwen-coach-v1 --judge anthropic \
    --out eval/reports/v1_finetuned.json
```

Compare against `eval/reports/baseline_29g_2026-07-10.json` — **beat factual 2.90 / overall 3.38**,
and check whether `clean_draw` grounding climbs from the baseline's 0.38. That delta is the
whole point of the moat.